# SemanticEmbed: AI Agent Pipeline Analysis

Find structural risks in LLM orchestration graphs -- vendor concentration, gateway bottlenecks, and guardrail single points of failure.

[GitHub](https://github.com/jmurray10/semanticembed-sdk)

In [ ]:
!pip install -q semanticembed
from semanticembed import encode, report
print("Ready.")

---
## The Problem

AI agent architectures have the same structural risks as microservice architectures -- SPOFs, amplification cascades, convergence bottlenecks -- but they're harder to see because the "services" are LLM endpoints, gateways, and guardrails instead of containers.

Runtime observability tells you your agent is slow. 6D structural analysis tells you **why it's structurally fragile** -- from the call graph alone.

---
## Example 1: Multi-Agent System with Vendor Concentration

A production AI system with a supervisor, 4 specialist agents, guardrails, and a RAG pipeline. All agents call the same LLM provider.

In [ ]:
# Production multi-agent system
edges = [
    # Supervisor orchestrates agents
    ("supervisor",       "planner-agent"),
    ("supervisor",       "research-agent"),
    ("supervisor",       "code-agent"),
    ("supervisor",       "writer-agent"),
    
    # All agents call the same LLM provider
    ("planner-agent",    "openai-gpt4"),
    ("research-agent",   "openai-gpt4"),
    ("code-agent",       "openai-gpt4"),
    ("writer-agent",     "openai-gpt4"),
    
    # Guardrails on the same provider
    ("supervisor",       "input-guardrail"),
    ("supervisor",       "output-guardrail"),
    ("input-guardrail",  "openai-moderation"),
    ("output-guardrail", "openai-moderation"),
    
    # RAG pipeline
    ("research-agent",   "rag-retriever"),
    ("rag-retriever",    "embedding-api"),
    ("embedding-api",    "vector-db"),
]

result = encode(edges)
print(result.table)
print()
print(report(result))

---
## Reading the Results

Look for these patterns in agent architectures:

| Pattern | What to look for | Risk |
|---------|-----------------|------|
| **Vendor concentration** | Single LLM node with high in-degree | OpenAI outage = total system failure |
| **Gateway bottleneck** | Gateway node with high fanout + high criticality | Gateway failure cuts off all models |
| **RAG chain depth** | 3-4 hop chain with no parallel path | Silent quality degradation |
| **Guardrail SPOF** | Guardrail at max depth with no redundancy | Every response depends on one check |

In [ ]:
# Inspect the vendor concentration risk
dim_names = ["depth", "independence", "hierarchy", "throughput", "criticality", "fanout"]

print("VENDOR CONCENTRATION ANALYSIS")
print("=" * 50)
for node in ["openai-gpt4", "openai-moderation", "vector-db"]:
    v = result.vectors[node]
    print(f"\n{node}:")
    for name, val in zip(dim_names, v):
        bar = "#" * int(val * 30)
        print(f"  {name:<15} {val:.3f}  {bar}")

---
## Example 2: AI Gateway Architecture

An agent that routes through a gateway (like Kong AI Gateway) to multiple LLM providers. The gateway is a structural chokepoint.

In [ ]:
# Agent -> Gateway -> Multiple providers
gateway_edges = [
    ("agent",           "ai-gateway"),
    ("ai-gateway",      "openai/gpt-4o"),
    ("ai-gateway",      "openai/gpt-4o-mini"),
    ("ai-gateway",      "anthropic/claude-sonnet"),
    ("ai-gateway",      "aws/bedrock-llama"),
    ("ai-gateway",      "local/ollama-mistral"),
    
    # Bedrock models have guardrails
    ("aws/bedrock-llama", "bedrock-guardrails"),
]

result_gw = encode(gateway_edges)
print(result_gw.table)
print()
print(report(result_gw))
print()
print("Key insight: ai-gateway has the highest fanout -- it's the amplification point.")
print("If the gateway fails, all 5 model providers are unreachable simultaneously.")

---
## Example 3: Fix the Risk â€” Add Redundancy

What if the agent can call some models directly, bypassing the gateway?

In [ ]:
from semanticembed import drift

# Add direct fallback paths for critical models
gateway_edges_fixed = gateway_edges + [
    ("agent", "openai/gpt-4o"),          # direct fallback
    ("agent", "anthropic/claude-sonnet"),  # direct fallback
]

result_fixed = encode(gateway_edges_fixed)
print("AFTER ADDING DIRECT FALLBACK PATHS")
print("=" * 50)
print(result_fixed.table)
print()
print(report(result_fixed))
print()

# Show what changed
changes = drift(result_gw, result_fixed)
print("\nSTRUCTURAL DRIFT:")
for node, deltas in changes.items():
    print(f"  {node}:")
    for dim, delta in deltas.items():
        direction = "+" if delta > 0 else ""
        print(f"    {dim}: {direction}{delta:.4f}")

---
## Your Own Agent Graph

Replace the edges below with your agent architecture. Any directed graph works -- agents, models, tools, gateways, guardrails, databases.

In [ ]:
# Replace with your own agent pipeline
my_agent_edges = [
    ("orchestrator", "agent-1"),
    ("orchestrator", "agent-2"),
    ("agent-1",      "llm-provider"),
    ("agent-2",      "llm-provider"),
    ("agent-1",      "tool-api"),
]

my_result = encode(my_agent_edges)
print(my_result.table)
print()
print(report(my_result))

---

**Next:** [03 - Drift Detection](https://colab.research.google.com/github/jmurray10/semanticembed-sdk/blob/main/notebooks/03_drift_detection.ipynb) -- compare two graph versions

*Patent pending. Application #63/994,075.*

## Parse real LangGraph / CrewAI / AutoGen code

If your agents are defined with one of the popular Python frameworks, the SDK
parses the actual graph definitions via AST -- no need to install the framework
to extract from it.

```python
import semanticembed as se

# LangGraph workflow file
edges = se.extract.from_langgraph("workflow.py")

# CrewAI script with Crew(manager_agent=...) fan-out
edges = se.extract.from_crewai("crew.py")

# AutoGen GroupChat (with or without an explicit GroupChatManager)
edges = se.extract.from_autogen("agents.py")

result = se.encode(edges)
print(result.table)
```

`extract.from_directory()` picks these up automatically by scanning Python
files for the relevant `import` statements.
